In [ ]:
#Imports ?


**Calculate seasonal anomalies of precip**

In [ ]:
#Code ?

**Calculate annual anomalies of precip**

In [ ]:
#Code ?

**Plotting the time-series of precipitation anomalies (annual and seasonal)**

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np 
from scipy.stats import linregress 
# Subset the required seasons 
son_df = seasonal_mean.xs('SON', level='season') 
djf_df = seasonal_mean.xs('DJF', level='season') 
mam_df = seasonal_mean.xs('MAM', level='season') 
# Function to calculate summary stats for plotting
def compute_stats(df):
    df_stats = pd.DataFrame({
        'mean': df.mean(axis=1),
        'p5': df.quantile(0.05, axis=1),
        'p95': df.quantile(0.95, axis=1),
        'min': df.min(axis=1),
        'max': df.max(axis=1)
    })
    df_stats.index.name = 'season_year'
    df_stats.reset_index(inplace=True)
    return df_stats

# Compute stats for each season
ann_stats = compute_stats(annual_mean)
son_stats = compute_stats(son_df)
djf_stats = compute_stats(djf_df)
mam_stats = compute_stats(mam_df)

# List of all stats and labels
panels = [('ANN', ann_stats), ('SON', son_stats), ('DJF', djf_stats), ('MAM', mam_stats)]

# === PLOT ===
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for i, (title, stats) in enumerate(panels):
    ax = axes[i]
    # Plot mean line
    ax.plot(stats['season_year'], stats['mean'], color='black', label='Mean SPEI')
    ax.fill_between(stats['season_year'], stats['p5'], stats['p95'], color='orange', alpha=0.3, label='5th–95th Percentile')
    ax.plot(stats['season_year'], stats['min'], linestyle='--', color='black', label='Min–Max Boundaries' if i == 0 else "")
    ax.plot(stats['season_year'], stats['max'], linestyle='--', color='black')

     # === Trend line calculation ===
    x = stats['season_year'].values
    y = stats['mean'].values
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    trend_per_decade = slope * 10

    # Plot the trend line
    ax.plot(x, intercept + slope * x, color='blue', linestyle='-', linewidth=1.2, label='Trend line')

    # Add text for trend and p-value
    ax.text(
        x.min(), -2.7,
        f"Trend = {trend_per_decade:.3f}/decade\np = {p_value:.3f}",
        fontsize=9,
        color='blue',
        va='top'
    )

    # X-axis ticks: marks every year, label every 3 years
    years = np.arange(x.min(), x.max() + 1)
    year_labels = [str(y) if y % 3 == 0 else "" for y in years]
    ax.set_xticks(years)
    ax.set_xticklabels(year_labels, rotation=0)

    # Highlight 5 driest years
    lowest = stats.nsmallest(5, 'mean')
    ax.scatter(lowest['season_year'], lowest['mean'], color='red', zorder=5)
    for _, row in lowest.iterrows():
        ax.text(row['season_year'], row['mean'] - 0.6, str(int(row['season_year'])), color='red', fontsize=9, ha='center', rotation=90)

    ax.set_title(title, fontdict={'weight':'bold'})
    if i % 2 == 0:
        ax.set_ylabel("SPEI12", fontdict={'weight':'bold'})
    ax.grid(axis='y')

# Add legend only once
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.subplots_adjust(bottom=0.05)
plt.show()